# 条件类型与 infer

学习目标：能按类型关系选择结果类型，用 infer 提取结构，并辨别联合分布、never、重载与递归的边界。

前置知识：泛型约束、联合、never、数组与元组、函数签名、索引访问类型。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/14-conditional-types/。

1. [main.ts](scripts/14-conditional-types/main.ts)：配套实现与示例。
2. [tsconfig.json](scripts/14-conditional-types/tsconfig.json)：本章独立项目配置。
3. [type-errors.ts](scripts/14-conditional-types/type-errors.ts)、[tsconfig.errors.json](scripts/14-conditional-types/tsconfig.errors.json)：单独检查的类型反例。

Step 1：检查本章正常示例的类型。

```bash
npm run check:14
```

Step 2：生成本章 JavaScript。

```bash
npm run build:14
```

Step 3：运行本章正常示例。

```bash
npm run run:14
# 正常退出；各段预期输出见代码注释。
```

正常片段均按正文顺序节选自 main.ts，前文定义在后续片段中继续使用。直接运行完整项目；反例使用独立配置，不进入正常运行入口。

## 1 条件判断与泛型约束

条件类型（conditional type）在类型检查时选择结果：T extends U ? X : Y 中，T 是待判断类型，U 是比较目标，X、Y 分别是真分支和假分支的结果类型；extends 检查可赋值关系，不是运行时 instanceof。

泛型参数列表中的 extends 会拒绝不满足约束的实参。放在条件类型中则允许输入进入判断并得到假分支，两种用途不能混为一谈。

```typescript
export type Constrained<T extends { code: number }> = T["code"];
export type CodeOf<T> = T extends { code: number } ? T["code"] : "absent";
const present: CodeOf<{ code: 201 }> = 201;
const absent: CodeOf<{ title: string }> = "absent";
console.log(present, absent);
// 预期输出：201 absent
```

## 2 infer 从容器和函数提取类型

infer 在条件的结构匹配中引入待推断类型，只有匹配成功后的真分支能使用它。下面 Item 代表元素类型；Args 代表参数元组；Result 代表返回值类型。

readonly 数组模式同时覆盖可变数组和只读数组。函数模式用推断参数元组表示所需结构，不要求实际调用函数，更不会执行异步工作。

```typescript
export type ElementOf<T> = T extends readonly (infer Item)[] ? Item : never;
export type ResultOf<T> = T extends (...args: infer Args) => infer Result ? Result : never;
type ArgsOf<T> = T extends (...args: infer Args) => unknown ? Args : never;
const element: ElementOf<readonly number[]> = 3;
function label(code: number, suffix: string) { return `${code}:${suffix}`; }
const args: ArgsOf<typeof label> = [7, "ok"];
const result: ResultOf<typeof label> = label(...args);
console.log(element, result);
// 预期输出：3 7:ok
```

## 3 联合类型何时分布

条件左侧是直接出现的类型参数 T 时，传入联合通常逐个成员判断，称为分布式条件类型（distributive conditional type）。下面 OneArray 先分别处理 string 与 number，最后合并结果。

把 T 放进单元素元组后，判断的是整个元组，可阻止这种分布。于是 WholeArray 的数组元素允许混合，而 OneArray 要求整个数组属于某个分支。不是只要出现联合就一定分布。

```typescript
export type OneArray<T> = T extends unknown ? T[] : never;
export type WholeArray<T> = [T] extends [unknown] ? T[] : never;
const single: OneArray<string | number> = [1, 2];
const mixed: WholeArray<string | number> = [1, "二"];
console.log(JSON.stringify(single), JSON.stringify(mixed));
// 预期输出：[1,2] [1,"二"]
```

## 4 never 与过滤

never 没有可取值，在分布式条件类型中相当于没有成员可以参与计算，因此可能直接得到 never，而不是直觉中的某个分支。判断一个类型是否整体为 never，需要像 IsNever 那样阻止分布。

过滤类型把不需要的联合成员变为 never，剩余成员合并后构成结果。它只改变静态类型，不会筛选运行时数组。

```typescript
export type KeepText<T> = T extends string ? T : never;
type IsNever<T> = [T] extends [never] ? true : false;
const word: KeepText<string | number> = "保留";
const empty: IsNever<KeepText<never>> = true;
console.log(word, empty);
// 预期输出：保留 true
```

## 5 重载推断采用最后的签名

对多个调用签名的函数做条件推断，使用最后一个可见重载签名；不会像真实调用那样按某组实参重新选择重载。实现签名用于检查实现，不能据此认为它自动成为对外可见的兜底重载。

下面显式提供联合输入、联合输出的末尾重载，ResultOf 得到 string 与 number 的联合；具体调用 flip(2) 仍按匹配的重载得到 string。

```typescript
export function flip(value: number): string;
export function flip(value: string): number;
export function flip(value: string | number): string | number;
export function flip(value: string | number): string | number {
  return typeof value === "number" ? String(value) : value.length;
}
const extracted: ResultOf<typeof flip> = 5;
const called: string = flip(2);
console.log(extracted, called);
// 预期输出：5 2
```

## 6 递归条件类型与终止分支

递归条件类型让结果继续引用自身，适合表达有限层的嵌套结构。Leaf 中 T 是当前层，Item 是剥去数组一层后的元素；遇到非数组便返回 T，构成终止条件。

类型递归不执行数组展平。层数和联合分支过多会增加检查成本，还可能碰到编译器递归限制；不应把复杂类型运算当作运行算法替代品，也不依赖固定的内部深度上限。

```typescript
type Leaf<T> = T extends readonly (infer Item)[] ? Leaf<Item> : T;
const leaf: Leaf<readonly (readonly number[])[]> = 9;
console.log(leaf);
// 预期输出：9
```

## 7 检查类型边界

下面的 [type-errors.ts](scripts/14-conditional-types/type-errors.ts) 只用于检查，不执行。逐项阅读注释，修正时保留原本需求，不通过断言或关闭检查掩盖错误。

```typescript
import type { Constrained, ElementOf, OneArray, KeepText } from "./main.js";
type MissingCode = Constrained<{ title: string }>; // 约束处就拒绝输入，不进入条件分支。
const wrongElement: ElementOf<readonly number[]> = "3"; // 提取的是 number。
const mixed: OneArray<string | number> = [1, "二"]; // 不符合 string[] 或 number[]。
const impossible: KeepText<never> = "有值"; // 结果为 never，不能赋普通值。
// 预期诊断包含：TS2741, TS2322。
```

Step 1：单独检查反例并对照错误位置与原因。

```bash
npm run errors:14
# 本章固定编译器预期退出码为 1；正常项目命令的退出码为 0。
```

## 本章小结

泛型约束限制输入，条件类型选择结果。infer 从匹配结构推断类型；裸类型参数触发联合分布，元组包装可以阻止分布。重载提取与调用选择不同，递归必须有清楚边界。

## 练习

1. 定义 ValueOfBox\<T\>，提取 { value: 某类型 }，其他输入得到 never；用 string 与不含 value 的对象分别验证。

2. 写出 OneArray\<string | number\> 与 WholeArray\<string | number\> 的合法值；确认只有后者接受 [1, "二"]。

3. 给 Leaf 增加三层只读数组的类型实参；核对最终仍接受数值，并解释运行时数组为什么没有被展开。

## 参考与引用来源

- TypeScript 官方文档：[Conditional Types：约束、infer、重载与分布](https://www.typescriptlang.org/docs/handbook/2/conditional-types.html)；[2.8：Distributive conditional types](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-2-8.html#distributive-conditional-types)；[4.1：Recursive Conditional Types](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-4-1.html#recursive-conditional-types)；[Overload Signatures and the Implementation Signature](https://www.typescriptlang.org/docs/handbook/2/functions.html#overload-signatures-and-the-implementation-signature)。